# YOLO 车牌检测 + CRNN 车牌识别（Kaggle 完整版）

整个项目分 4 部分，按顺序从上往下跑：

| 部分 | 做什么 | 用什么 | 大约时间（T4） |
|---|---|---|---|
| 第 1 天 | 从 CCPD 生成检测数据和识别数据 | CPU | 15–25 分钟 |
| 第 2 天 | 训练 YOLOv8n 找车牌，在测试集上评估 | GPU | 1.5–2.5 小时 |
| 第 3 天 | PaddleOCR 基线 + 训练自己的 CRNN 读车牌号 | GPU | 30–60 分钟 |
| 第 4 天 | 端到端评测、OpenCV 传统方法对比、困难场景测试 | GPU | 10–20 分钟 |

## 怎么用

1. **右侧面板**：Accelerator 选 **GPU T4 x2**，Internet 打开，Input 里加好 `ccpd-preprocess` 数据集。
2. **先试跑**：下面「总开关」里保持 `QUICK_TEST = True`，点 **Run All**。每个模型只训 1 轮，30–40 分钟能跑完，目的是确认整条流程没有报错。
3. **正式跑**：把 `QUICK_TEST` 改成 `False`，点右上角 **Save Version → Save & Run All (Commit)**。它会在后台跑完全部（约 3–4 小时），关掉浏览器也没关系。
4. **拿结果**：跑完后打开这个版本的 **Output**，下载 `results.zip`。所有关键数字都在 `results.txt` 里。

> **防卡死：** 每一步都加了时间上限（`timeout`），某一步卡住会被自动停掉、继续下一步，整个流程最多 9 小时左右，不会超过 Kaggle 的 12 小时限制。每一步开始和结束都会打印时间，出问题时看日志就知道卡在哪。

In [ ]:
# ===== 第 0 步：检查环境 =====
!nvidia-smi -L
!pip install -q ultralytics

import torch, numpy, ultralytics
print("torch", torch.__version__, "| numpy", numpy.__version__, "| ultralytics", ultralytics.__version__)
print("GPU 可用：", torch.cuda.is_available())

In [ ]:
# ===== 总开关 =====
QUICK_TEST = False   # True：试跑，每个模型只训 1 轮，用来检查流程有没有报错
                     # False：正式训练（YOLO 50 轮，CRNN 30 轮）

YOLO_EPOCHS = 1 if QUICK_TEST else 50
CRNN_EPOCHS = 1 if QUICK_TEST else 30
print(f"{'试跑' if QUICK_TEST else '正式训练'}：YOLO {YOLO_EPOCHS} 轮，CRNN {CRNN_EPOCHS} 轮")

# 所有结果都会追加到 results.txt，先清空上一次的
import os
import time
os.chdir("/kaggle/working")
if os.path.exists("results.txt"):
    os.remove("results.txt")


def log(message):
    """打印带时间的进度。如果哪一步卡住，看日志里最后一行的时间就知道停在哪"""
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


log("开始运行")

---
# 第 1 天：生成训练数据

CCPD 不需要自己标注：**车牌位置和车牌号都写在文件名里**。比如：

```
00205459770115-90_85-352,516_448,547-444,547_368,549_364,517_440,515-0_0_22_10_26_29_24-128-7.jpg
```

用 `-` 切开后：第 3 段 `352,516_448,547` 是车牌框（左上角、右下角），第 5 段 `0_0_22_10_26_29_24` 是 7 个字符的编号，查表就是「皖AYL250」。

`prepare_data.py` 做两件事：
- **检测数据**（给 YOLO）：复制原图 + 写一个标签 txt，告诉 YOLO 车牌在哪
- **识别数据**（给 CRNN）：裁出车牌小图 + 记下车牌号

图片在 Kaggle 的网络盘上，一张一张读很慢，所以用 **16 个线程同时读**。

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py —— 把 CCPD 数据集整理成两份训练数据

  1. 检测数据（给 YOLO 用）：plate/images/ 放原图，plate/labels/ 放同名标签 txt
  2. 识别数据（给 CRNN 用）：plate/rec/ 放裁好的车牌小图，rec/train.txt 等记录「图片路径<Tab>车牌号」

这个文件里的 parse_name、crop_plate、iou、save_result 后面几个脚本也会导入使用。
"""
import os
import re
import random
import shutil
from concurrent.futures import ThreadPoolExecutor

import cv2


# ==================================================
# 第一部分：设置（想改路径或数量，只改这里）
# ==================================================

CCPD_DIR = "/kaggle/input/datasets/binh234/ccpd-preprocess/CCPD2019"
SOURCE_DIR = f"{CCPD_DIR}/ccpd_base"
OUTPUT_DIR = "/kaggle/working/plate"
RESULT_FILE = "/kaggle/working/results.txt"

NUM_TRAIN = 8000         # 检测训练集
NUM_VAL = 1000           # 检测验证集
NUM_TEST = 1000          # 检测测试集
NUM_EXTRA_REC = 40000    # 额外给识别模型的图片（只裁车牌，不占多少空间）
NUM_THREADS = 16         # 同时读多少张图


# ==================================================
# 第二部分：车牌字符表
# 文件名里的车牌号是数字编号，要用这三张表翻译成汉字和字母
# ==================================================

# 第 1 位：省份
PROVINCES = ["皖", "沪", "津", "渝", "冀", "晋", "蒙", "辽", "吉", "黑", "苏", "浙",
             "京", "闽", "赣", "鲁", "豫", "鄂", "湘", "粤", "桂", "琼", "川", "贵",
             "云", "藏", "陕", "甘", "青", "宁", "新", "警", "学", "O"]

# 第 2 位：字母（车牌里没有 I 和 O，最后的 "O" 是占位符）
LETTERS = ["A", "B", "C", "D", "E", "F", "G", "H", "J", "K", "L", "M",
           "N", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z", "O"]

# 第 3~7 位：字母或数字
LETTERS_AND_DIGITS = LETTERS[:-1] + ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "O"]


# ==================================================
# 第三部分：工具函数
# ==================================================

def parse_name(filename):
    """
    从 CCPD 文件名中读出车牌框和车牌号。

    用 "-" 切开后：
        第 3 段 "352,516_448,547"      → 车牌框：左上角 (352, 516)，右下角 (448, 547)
        第 5 段 "0_0_22_10_26_29_24"   → 车牌号的 7 个字符编号

    返回：((left, top, right, bottom), "皖AYL250")
    """
    name = os.path.basename(filename)   # 去掉前面的文件夹路径
    name = name[:-4]                    # 去掉结尾的 ".jpg"
    parts = name.split("-")

    # --- 车牌框 ---
    # 不同版本的 CCPD 用不同符号分隔坐标（& _ ,），三种都当作分隔符
    box_numbers = re.split(r"[&_,]", parts[2])
    left, top, right, bottom = [int(n) for n in box_numbers]
    box = (left, top, right, bottom)

    # --- 车牌号 ---
    char_ids = [int(n) for n in parts[4].split("_")]
    province = PROVINCES[char_ids[0]]
    letter = LETTERS[char_ids[1]]
    rest = ""
    for char_id in char_ids[2:]:
        rest += LETTERS_AND_DIGITS[char_id]
    plate = province + letter + rest

    return box, plate


def box_to_yolo_label(box, img_width, img_height):
    """
    把车牌框换成 YOLO 标签格式：  类别  中心x  中心y  宽  高
    后四个数都要除以图片宽高，变成 0~1 之间的比例。只有车牌一类，类别固定是 0。
    """
    left, top, right, bottom = box

    center_x = (left + right) / 2 / img_width
    center_y = (top + bottom) / 2 / img_height
    box_width = (right - left) / img_width
    box_height = (bottom - top) / img_height

    return f"0 {center_x:.6f} {center_y:.6f} {box_width:.6f} {box_height:.6f}"


def crop_plate(img, box, pad=0.08):
    """
    按车牌框把车牌裁出来，四周各多留 8% 的边。
    训练和评测都用这个函数，保证裁法一致。
    """
    left, top, right, bottom = box
    plate_width = right - left
    plate_height = bottom - top
    img_height, img_width = img.shape[:2]

    # 往外扩一点，同时不能超出图片边界
    left = max(0, int(left - pad * plate_width))
    top = max(0, int(top - pad * plate_height))
    right = min(img_width, int(right + pad * plate_width))
    bottom = min(img_height, int(bottom + pad * plate_height))

    return img[top:bottom, left:right]


def iou(box_a, box_b):
    """
    交并比 IoU = 两个框重叠的面积 / 两个框合起来的面积。
    1 表示完全重合，0 表示完全不挨着。用来判断定位准不准。
    """
    overlap_width = max(0, min(box_a[2], box_b[2]) - max(box_a[0], box_b[0]))
    overlap_height = max(0, min(box_a[3], box_b[3]) - max(box_a[1], box_b[1]))
    overlap_area = overlap_width * overlap_height

    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - overlap_area

    return overlap_area / union_area


def run_in_threads(func, jobs, title):
    """
    用多线程对每个 job 调用 func(*job)，按原来的顺序返回结果，并打印进度。
    图片在网络盘上，大部分时间都在等文件，多个线程一起等能快很多。
    """
    results = []
    with ThreadPoolExecutor(max_workers=NUM_THREADS) as pool:
        for n, result in enumerate(pool.map(lambda job: func(*job), jobs), start=1):
            results.append(result)
            if n % 5000 == 0 or n == len(jobs):
                print(f"  {title}: {n}/{len(jobs)}", flush=True)
    return results


def save_result(text):
    """打印一行结果，同时追加到 results.txt，最后统一查看"""
    print(text)
    with open(RESULT_FILE, "a", encoding="utf-8") as f:
        f.write(text + "\n")


# ==================================================
# 第四部分：生成数据
# ==================================================

def split_files():
    """读出所有图片名，打乱后切成 训练 / 验证 / 测试 / 额外识别 四份，互不重叠"""
    all_files = sorted(f for f in os.listdir(SOURCE_DIR) if f.endswith(".jpg"))
    random.seed(0)                      # 固定随机种子，每次切出来的都一样
    random.shuffle(all_files)
    print(f"ccpd_base 共 {len(all_files)} 张")

    end_train = NUM_TRAIN
    end_val = end_train + NUM_VAL
    end_test = end_val + NUM_TEST
    end_extra = end_test + NUM_EXTRA_REC

    return {
        "train": all_files[:end_train],
        "val": all_files[end_train:end_val],
        "test": all_files[end_val:end_test],
        "extra": all_files[end_test:end_extra],
    }


def make_detection_sample(split, filename):
    """处理一张检测图：复制原图 + 写 YOLO 标签"""
    image_path = f"{OUTPUT_DIR}/images/{split}/{filename}"
    shutil.copy(f"{SOURCE_DIR}/{filename}", image_path)

    img = cv2.imread(image_path)        # 读刚复制到本地的那份，比读网络盘快
    img_height, img_width = img.shape[:2]
    box, _ = parse_name(filename)       # 检测只需要框，车牌号用 _ 忽略
    label = box_to_yolo_label(box, img_width, img_height)

    label_path = f"{OUTPUT_DIR}/labels/{split}/{filename.replace('.jpg', '.txt')}"
    with open(label_path, "w") as f:
        f.write(label + "\n")


def make_recognition_sample(split, index, filename):
    """处理一张识别图：裁出车牌小图保存，返回「图片路径<Tab>车牌号」"""
    box, plate = parse_name(filename)
    img = cv2.imread(f"{SOURCE_DIR}/{filename}")
    crop = crop_plate(img, box)

    crop_name = f"{split}/{index:06d}.jpg"     # train/000000.jpg, train/000001.jpg, ...
    cv2.imwrite(f"{OUTPUT_DIR}/rec/{crop_name}", crop)
    return f"{crop_name}\t{plate}"


def main():
    files = split_files()

    # 1. 检测数据
    for split in ["train", "val", "test"]:
        os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)
        jobs = [(split, filename) for filename in files[split]]
        run_in_threads(make_detection_sample, jobs, f"[检测] {split}")

    # 2. 识别数据（训练集多加 4 万张，识别模型需要更多样本）
    rec_files = {
        "train": files["train"] + files["extra"],
        "val": files["val"],
        "test": files["test"],
    }
    for split, filenames in rec_files.items():
        os.makedirs(f"{OUTPUT_DIR}/rec/{split}", exist_ok=True)
        jobs = [(split, i, filename) for i, filename in enumerate(filenames)]
        lines = run_in_threads(make_recognition_sample, jobs, f"[识别] {split}")
        with open(f"{OUTPUT_DIR}/rec/{split}.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")

    print("全部完成！")


# 直接运行这个文件时才执行 main()；被别的脚本 import 时不执行
if __name__ == "__main__":
    main()

In [ ]:
%%writefile ccpd.yaml
path: /kaggle/working/plate
train: images/train
val: images/val
test: images/test
names:
  0: plate

In [ ]:
# 先用一张图测试解析函数（几秒钟），没问题再正式生成
from prepare_data import parse_name, box_to_yolo_label, SOURCE_DIR
filename = sorted(os.listdir(SOURCE_DIR))[0]
box, plate = parse_name(filename)
print("文件名：", filename)
print("车牌框：", box, "| 车牌号：", plate)
print("YOLO 标签：", box_to_yolo_label(box, 720, 1160))

In [ ]:
# 正式生成数据（约 15–25 分钟）
# timeout 3600：最多跑 1 小时，超时自动停止。python -u：输出不缓存，日志里能实时看到进度
log("第 1 天：开始生成数据")
!timeout 3600 python -u prepare_data.py
log("数据生成结束")

# 检查数量
for name, folder, expected in [("检测训练图", "plate/images/train", 8000),
                               ("检测标签", "plate/labels/train", 8000),
                               ("识别训练图", "plate/rec/train", 48000)]:
    print(f"{name}：{len(os.listdir(folder))}（应为 {expected}）")
!head -3 plate/rec/train.txt

---
# 第 2 天：训练 YOLOv8n 找车牌

`yolov8n.pt` 是在 COCO（80 类常见物体）上预训练好的最小号 YOLO。我们在它的基础上**微调**，让它只学会找「车牌」这一类。

训练时每一轮结束会打印验证集指标，重点看 **mAP50** 这一列，正常会很快升到 0.95 以上。

In [ ]:
# 用命令行训练，前面加 timeout 10800：最多 3 小时，超时自动停下
# 每一轮结束都会保存权重（best.pt / last.pt），就算被停下，已经训练的部分也不会白费
#   model=yolov8n.pt  第一次会自动下载预训练权重
#   workers=4         Kaggle 只有 4 核 CPU
#   exist_ok=True     重复训练时覆盖 runs/det，路径不会变成 det2、det3
log("第 2 天：开始训练 YOLO")
!timeout 10800 yolo detect train data=/kaggle/working/ccpd.yaml model=yolov8n.pt epochs={YOLO_EPOCHS} imgsz=640 batch=32 workers=4 device=0 project=/kaggle/working/runs name=det exist_ok=True
log("YOLO 训练结束")

In [ ]:
# 在测试集上评估（测试集训练时完全没见过）
from ultralytics import YOLO
from prepare_data import save_result

log("开始在测试集上评估 YOLO")

best_det = YOLO("/kaggle/working/runs/det/weights/best.pt")
metrics = best_det.val(
    data="/kaggle/working/ccpd.yaml",
    split="test",
    project="/kaggle/working/runs",
    name="det_test",
    exist_ok=True,
)

save_result("【YOLO 检测 · 测试集】")
save_result(f"  Precision  {metrics.box.mp:.4f}   （报出的框里有多少是对的）")
save_result(f"  Recall     {metrics.box.mr:.4f}   （真实车牌里有多少被找到了）")
save_result(f"  mAP50      {metrics.box.map50:.4f}   （简历上写这个）")
save_result(f"  mAP50-95   {metrics.box.map:.4f}   （更严格的标准）")

---
# 第 3 天：车牌识别

先测**通用 OCR**（PaddleOCR）直接读车牌有多准，作为基线；再用 PyTorch 自己训练一个专门读车牌的 **CRNN**。两个数一对比，就是「识别准确率由 X% 提升至 Y%」。

## 3.1 PaddleOCR 基线

PaddleOCR 和 PyTorch 装在一起容易版本冲突，所以把它**单独装到 `/tmp/pd` 文件夹**，运行时用 `PYTHONPATH` 指过去，不影响其他步骤。只识别 1000 张小图，几分钟就够。

> Kaggle 现在是 Python 3.13，只能装 PaddlePaddle 3.x，所以用的是 PaddleOCR 3.x 的新接口 `TextRecognition`（默认是 PP-OCRv5 识别模型）。
>
> 如果安装失败，这一步会打印「PaddleOCR 基线：跳过」，不影响后面的步骤。

In [ ]:
%%writefile baseline_paddle.py
# baseline_paddle.py —— 直接用 PaddleOCR 通用识别模型读车牌截图，作为基线
from paddleocr import TextRecognition

ROOT = "/kaggle/working/plate/rec"

with open(f"{ROOT}/test.txt", encoding="utf-8") as f:
    items = [line.rstrip("\n").split("\t") for line in f if line.strip()]

model = TextRecognition()     # 默认 PP-OCRv5 识别模型，第一次运行会自动下载
image_paths = [f"{ROOT}/{path}" for path, _ in items]
results = model.predict(input=image_paths, batch_size=32)   # 按输入顺序返回，每个结果里有 rec_text

right = 0
for (path, true_plate), result in zip(items, results):
    pred = result["rec_text"].replace("·", "").replace(" ", "").upper()   # 去掉车牌中间的点和空格
    right += pred == true_plate

text = f"【PaddleOCR 基线】识别准确率（真实框截图）：{right / len(items):.2%}"
print(text)
with open("/kaggle/working/results.txt", "a", encoding="utf-8") as f:
    f.write(text + "\n")

In [ ]:
# 装到单独的文件夹，不影响 PyTorch 环境（安装过程的提示很多，全部隐藏掉）
# 安装最多 15 分钟、识别最多 20 分钟，超时就跳过这一步
log("第 3 天：安装 PaddleOCR")
!timeout 900 pip install -q --target /tmp/pd paddlepaddle==3.2.0 "paddleocr>=3,<4" > /dev/null 2>&1
log("开始 PaddleOCR 基线识别")
!PYTHONPATH=/tmp/pd timeout 1200 python -u baseline_paddle.py || echo "【PaddleOCR 基线】跳过（安装或运行失败）" | tee -a results.txt
log("PaddleOCR 基线结束")

## 3.2 训练 CRNN + CTC

**CRNN 的思路：**
1. **CNN**（卷积网络）把 48×160 的车牌图压成 3×40 的特征图，相当于从左到右切成 **40 列**
2. **双向 LSTM** 让每一列都能看到左右两边的内容
3. **全连接层**给每一列预测一个字符

**CTC 的作用：** 40 列的预测怎么变成 7 个字符？CTC 引入一个「空白」符号，解码时**合并相邻重复的字符、去掉空白**。比如 `皖皖_AA_Y_L_2_5_0` → `皖AYL250`。这样训练时只需要整串车牌号，**不用标注每个字在图里的位置**。

> 前几轮准确率一直是 0% 很正常：CTC 刚开始常常只输出空白，一般到第 3–5 轮才开始上升。

In [ ]:
%%writefile train_crnn.py
"""
train_crnn.py —— 用 PyTorch 训练 CRNN + CTC 车牌识别模型
用法：python train_crnn.py 训练轮数
"""
import sys

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# OpenCV 自己也会开多线程，和 DataLoader 的多进程一起用时偶尔会互相等待、卡死。
# 关掉 OpenCV 的多线程就能避免（读小图本来也用不上）
cv2.setNumThreads(0)

ROOT = "/kaggle/working/plate/rec"
MODEL_PATH = "/kaggle/working/crnn_best.pt"

# 所有可能出现的字符。编号从 1 开始，0 留给 CTC 的「空白」
CHARS = ("皖沪津渝冀晋蒙辽吉黑苏浙京闽赣鲁豫鄂湘粤桂琼川贵云藏陕甘青宁新警学"
         "ABCDEFGHJKLMNOPQRSTUVWXYZ0123456789")
IMG_WIDTH, IMG_HEIGHT = 160, 48


# ==================================================
# 第一部分：数据
# ==================================================

def preprocess(img):
    """把车牌图缩放到 160x48，转成 3x48x160、数值在 0~1 之间的张量"""
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    tensor = torch.from_numpy(img).permute(2, 0, 1)   # 高x宽x通道 → 通道x高x宽
    return tensor.float() / 255.0


class PlateDataset(Dataset):
    """读 rec/train.txt 这样的列表，每次返回 (图片张量, 标签编号, 车牌号文字)"""

    def __init__(self, split, augment=False):
        with open(f"{ROOT}/{split}.txt", encoding="utf-8") as f:
            self.items = [line.rstrip("\n").split("\t") for line in f if line.strip()]
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, plate = self.items[i]
        img = cv2.imread(f"{ROOT}/{path}")

        if self.augment:   # 数据增强：随机调亮度，让模型适应不同光照
            brightness = np.random.uniform(0.6, 1.4)
            img = np.clip(img * brightness, 0, 255).astype(np.uint8)

        label = torch.tensor([CHARS.index(c) + 1 for c in plate])
        return preprocess(img), label, plate


def collate(batch):
    """把一批样本拼起来。CTC 要求所有标签首尾相连成一长串，再单独给出每个标签的长度"""
    imgs, labels, plates = zip(*batch)
    imgs = torch.stack(imgs)
    label_lengths = torch.tensor([len(label) for label in labels])
    labels = torch.cat(labels)
    return imgs, labels, label_lengths, list(plates)


# ==================================================
# 第二部分：模型
# ==================================================

def conv_block(in_channels, out_channels, pool=None):
    """卷积 + 批归一化 + ReLU，需要的话再加一个池化层缩小尺寸"""
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(pool))
    return layers


class CRNN(nn.Module):
    def __init__(self, num_classes=len(CHARS) + 1):   # +1 是 CTC 的空白
        super().__init__()
        self.cnn = nn.Sequential(
            *conv_block(3, 64, pool=(2, 2)),       # 48x160 → 24x80
            *conv_block(64, 128, pool=(2, 2)),     # → 12x40
            *conv_block(128, 256),
            *conv_block(256, 256, pool=(2, 1)),    # → 6x40（只压高度，保留宽度方向的 40 列）
            *conv_block(256, 512, pool=(2, 1)),    # → 3x40
        )
        self.rnn = nn.LSTM(512 * 3, 256, num_layers=2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256 * 2, num_classes)  # 双向 LSTM 输出是 256x2

    def forward(self, x):
        features = self.cnn(x)                       # B, 512, 3, 40
        b, c, h, w = features.shape
        features = features.permute(0, 3, 1, 2)      # B, 40, 512, 3：把宽度放到第 2 维
        features = features.reshape(b, w, c * h)     # B, 40, 1536：从左到右 40 个时间步
        output, _ = self.rnn(features)               # B, 40, 512
        return self.fc(output)                       # B, 40, 类别数


# ==================================================
# 第三部分：解码和评估
# ==================================================

def decode(logits):
    """CTC 贪心解码：每一列取概率最大的字符，合并相邻重复，去掉空白（编号 0）"""
    plates = []
    for sequence in logits.argmax(dim=2).tolist():
        chars = []
        previous = 0
        for k in sequence:
            if k != 0 and k != previous:
                chars.append(CHARS[k - 1])
            previous = k
        plates.append("".join(chars))
    return plates


@torch.no_grad()
def evaluate(model, loader, device):
    """整串车牌号完全正确才算对"""
    model.eval()
    right = total = 0
    for imgs, _, _, plates in loader:
        preds = decode(model(imgs.to(device)))
        right += sum(pred == plate for pred, plate in zip(preds, plates))
        total += len(plates)
    model.train()
    return right / total


# ==================================================
# 第四部分：训练
# ==================================================

def main():
    epochs = int(sys.argv[1]) if len(sys.argv) > 1 else 30
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 防卡死的两个设置：
    #   persistent_workers=True  读数据的子进程一直保留，不用每轮重新创建
    #   timeout=300              5 分钟还拿不到一批数据就直接报错，而不是一直卡着
    loader_args = dict(num_workers=2, persistent_workers=True, timeout=300, collate_fn=collate)
    train_loader = DataLoader(PlateDataset("train", augment=True), batch_size=128, shuffle=True, **loader_args)
    val_loader = DataLoader(PlateDataset("val"), batch_size=256, **loader_args)
    test_loader = DataLoader(PlateDataset("test"), batch_size=256, **loader_args)

    model = CRNN().to(device)
    ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)  # 学习率逐渐变小

    best_acc = -1
    for epoch in range(1, epochs + 1):
        for imgs, labels, label_lengths, _ in train_loader:
            logits = model(imgs.to(device))                        # B, T, C
            log_probs = logits.log_softmax(2).permute(1, 0, 2)     # T, B, C（CTCLoss 要求的形状）
            input_lengths = torch.full((imgs.size(0),), log_probs.size(0), dtype=torch.long)
            loss = ctc_loss(log_probs, labels.to(device), input_lengths, label_lengths)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

        val_acc = evaluate(model, val_loader, device)
        print(f"epoch {epoch}/{epochs}  loss {loss.item():.4f}  val_acc {val_acc:.2%}", flush=True)
        if val_acc > best_acc:                                     # 只保存验证集上最好的那一轮
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    test_acc = evaluate(model, test_loader, device)

    from prepare_data import save_result
    save_result(f"【CRNN 识别】测试集识别准确率（真实框截图）：{test_acc:.2%}")


if __name__ == "__main__":
    main()

In [ ]:
# 最多 2 小时。每一轮验证集最好的模型都会存到 crnn_best.pt，超时也不会全部白费
log("开始训练 CRNN")
!timeout 7200 python -u train_crnn.py {CRNN_EPOCHS}
log("CRNN 训练结束")

---
# 第 4 天：整体评测

今天产出简历上最关键的几个数字。**「定位正确」统一定义为：预测框和真实框的 IoU ≥ 0.7**（比常用的 0.5 更严格）。

1. **端到端**：YOLO 找车牌 → 裁剪 → CRNN 读字。认错的图会存进 `wrong_cases/`，用来做错误分析
2. **传统方法对比**：用 OpenCV（颜色 + 形态学 + 轮廓）定位车牌，和 YOLO 用同一个标准、同一批图比较
3. **困难场景**：从倾斜、恶劣天气、综合挑战三个子集各抽 500 张，看准确率掉多少

In [ ]:
%%writefile e2e_eval.py
"""
e2e_eval.py —— 端到端评测：YOLO 找车牌 → 裁剪 → CRNN 识别
用法：python e2e_eval.py 图片文件夹 场景名
"""
import os
import sys

import cv2
import torch
from ultralytics import YOLO

from prepare_data import parse_name, crop_plate, iou, save_result
from train_crnn import CRNN, preprocess, decode, MODEL_PATH

test_dir = sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/plate/images/test"
scene = sys.argv[2] if len(sys.argv) > 2 else "正常场景"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载两个训练好的模型
detector = YOLO("/kaggle/working/runs/det/weights/best.pt")
recognizer = CRNN().to(device).eval()
recognizer.load_state_dict(torch.load(MODEL_PATH, map_location=device))

wrong_dir = f"/kaggle/working/wrong_cases/{scene}"
os.makedirs(wrong_dir, exist_ok=True)

total = located = recognized = 0
for filename in sorted(os.listdir(test_dir)):
    true_box, true_plate = parse_name(filename)
    img = cv2.imread(os.path.join(test_dir, filename))
    total += 1

    # 1. 检测：取置信度最高的框；一个框都没找到，定位和识别都算错
    boxes = detector(img, verbose=False)[0].boxes
    if len(boxes) == 0:
        continue
    pred_box = boxes.xyxy[boxes.conf.argmax()].tolist()
    if iou(pred_box, true_box) >= 0.7:
        located += 1

    # 2. 裁剪 + 识别
    crop = crop_plate(img, pred_box)
    if crop.size == 0:
        continue
    with torch.no_grad():
        pred_plate = decode(recognizer(preprocess(crop).unsqueeze(0).to(device)))[0]

    if pred_plate == true_plate:
        recognized += 1
    else:   # 保存认错的图，文件名里写上「正确答案_pred_预测结果」
        cv2.imwrite(f"{wrong_dir}/{true_plate}_pred_{pred_plate}.jpg", crop)

save_result(f"【端到端 · {scene}】共 {total} 张 | "
            f"定位准确率（IoU≥0.7）{located / total:.2%} | 端到端识别准确率 {recognized / total:.2%}")

In [ ]:
%%writefile cv_baseline.py
"""
cv_baseline.py —— 传统方法定位车牌：蓝色区域 + 形态学闭运算 + 轮廓筛选
用法：python cv_baseline.py 图片文件夹
"""
import os
import sys

import cv2

from prepare_data import parse_name, iou, save_result


def locate(img):
    """返回找到的车牌框 (left, top, right, bottom)，找不到返回 None"""
    # 1. 找蓝色区域（CCPD 绝大多数是蓝牌）
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (100, 80, 60), (130, 255, 255))

    # 2. 闭运算：把字符之间的缝隙连起来，让整块车牌变成一个整体
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # 3. 找轮廓，按车牌的长宽比和最小面积筛选，留下面积最大的
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        looks_like_plate = 2 < w / h < 5.5 and w * h > 1500
        if looks_like_plate and (best is None or w * h > best[2] * best[3]):
            best = (x, y, w, h)

    if best is None:
        return None
    x, y, w, h = best
    return (x, y, x + w, y + h)


if __name__ == "__main__":
    test_dir = sys.argv[1] if len(sys.argv) > 1 else "/kaggle/working/plate/images/test"
    filenames = sorted(os.listdir(test_dir))

    located = 0
    for filename in filenames:
        pred_box = locate(cv2.imread(os.path.join(test_dir, filename)))
        true_box, _ = parse_name(filename)
        if pred_box is not None and iou(pred_box, true_box) >= 0.7:
            located += 1

    save_result(f"【OpenCV 传统方法 · 正常场景】共 {len(filenames)} 张 | "
                f"定位准确率（IoU≥0.7）{located / len(filenames):.2%}")

In [ ]:
%%writefile make_hard.py
"""
make_hard.py —— 从三个困难子集各随机抽 500 张，复制到 plate/hard/
"""
import os
import random
import shutil

from prepare_data import CCPD_DIR, OUTPUT_DIR, run_in_threads

random.seed(0)
for subset in ["tilt", "weather", "challenge"]:
    src_dir = f"{CCPD_DIR}/ccpd_{subset}"
    dst_dir = f"{OUTPUT_DIR}/hard/{subset}"
    if not os.path.isdir(src_dir):
        print(f"没有找到 {src_dir}，跳过")
        continue

    os.makedirs(dst_dir, exist_ok=True)
    all_files = sorted(f for f in os.listdir(src_dir) if f.endswith(".jpg"))
    chosen = random.sample(all_files, 500)
    jobs = [(f"{src_dir}/{f}", dst_dir) for f in chosen]
    run_in_threads(shutil.copy, jobs, f"[困难场景] {subset}")

In [ ]:
# 每个评测最多 30 分钟（正常几分钟就够）
# 1. 正常场景：端到端 + 传统方法对比（同一批 1000 张测试图）
log("第 4 天：端到端评测（正常场景）")
!timeout 1800 python -u e2e_eval.py /kaggle/working/plate/images/test 正常场景
log("OpenCV 传统方法对比")
!timeout 1800 python -u cv_baseline.py /kaggle/working/plate/images/test

# 2. 困难场景
log("准备困难场景数据")
!timeout 1800 python -u make_hard.py
for folder, scene in [("tilt", "倾斜"), ("weather", "恶劣天气"), ("challenge", "综合挑战")]:
    if os.path.isdir(f"/kaggle/working/plate/hard/{folder}"):
        log(f"端到端评测（{scene}）")
        !timeout 1800 python -u e2e_eval.py /kaggle/working/plate/hard/{folder} {scene}
log("评测全部结束")

---
# 汇总结果

下面会打印所有关键数字，并把需要下载的东西打包成 `results.zip`：

| 文件 | 用途 |
|---|---|
| `results.txt` | 所有准确率，发给 Claude 更新简历 |
| `runs/det/results.png`、`val_batch0_pred.jpg` | 训练曲线和预测效果图，放进 GitHub README |
| `runs/det/weights/best.pt`、`crnn_best.pt` | 训练好的两个模型 |
| `wrong_cases/` | 认错的车牌截图，用来做错误分析 |
| `*.py`、`ccpd.yaml` | 全部代码，上传 GitHub |

In [ ]:
print("=" * 70)
!cat results.txt
print("=" * 70)

!zip -qr results.zip results.txt runs/det/results.png runs/det/val_batch0_pred.jpg runs/det/weights/best.pt crnn_best.pt wrong_cases *.py ccpd.yaml
!ls -lh results.zip

# 正式运行时删掉生成的训练数据（3GB 左右，随时能重新生成），这样 Output 下载更快
if not QUICK_TEST:
    !rm -rf plate
log("全部完成")